# <span style="color:blue">PRÉ PROCESSAMENTO DE DADOS</span> #

## <span style="color:blue">PACOTES UTILIZADOS</span> ##

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
import kagglehub
from pathlib import Path
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import gc


/usr/local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## <span style="color:blue">OBTENÇÃO DE DADOS BRUTOS E TRANSFORMAÇÃO EM DATASET PRIMÁRIO</span> ## 

In [10]:
# ============================================================
# 01. CONFIGURAÇÕES
# ============================================================

dataset_kaggle = "kartik2112/fraud-detection"

pasta_dados = Path(
    "/projeto_tcc_2026/dados/dados_primarios"
)

arquivo_train = pasta_dados / "fraudTrain.csv"
arquivo_test = pasta_dados / "fraudTest.csv"

arquivo_dataset_primario = (
    pasta_dados / "dataset_primario.csv"
)


# ============================================================
# 02. GARANTIR QUE A PASTA DE DADOS EXISTA
# ============================================================

pasta_dados.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 03. VERIFICAR E BAIXAR fraudTrain.csv
# ============================================================

if not arquivo_train.exists():

    print(
        "fraudTrain.csv não encontrado."
    )

    print(
        "Baixando fraudTrain.csv do Kaggle..."
    )

    kagglehub.dataset_download(
        dataset_kaggle,
        path="fraudTrain.csv",
        output_dir=str(pasta_dados)
    )

    print(
        "fraudTrain.csv baixado com sucesso!"
    )

else:

    print(
        "fraudTrain.csv já existe."
    )

    print(
        "Download não necessário."
    )


# ============================================================
# 04. VERIFICAR E BAIXAR fraudTest.csv
# ============================================================

if not arquivo_test.exists():

    print(
        "\nfraudTest.csv não encontrado."
    )

    print(
        "Baixando fraudTest.csv do Kaggle..."
    )

    kagglehub.dataset_download(
        dataset_kaggle,
        path="fraudTest.csv",
        output_dir=str(pasta_dados)
    )

    print(
        "fraudTest.csv baixado com sucesso!"
    )

else:

    print(
        "\nfraudTest.csv já existe."
    )

    print(
        "Download não necessário."
    )


# ============================================================
# 05. CONFIRMAR SE OS DOIS ARQUIVOS EXISTEM
# ============================================================

if not arquivo_train.exists():

    raise FileNotFoundError(
        "fraudTrain.csv não foi encontrado."
    )

if not arquivo_test.exists():

    raise FileNotFoundError(
        "fraudTest.csv não foi encontrado."
    )


# ============================================================
# 06. CARREGAR AS DUAS BASES ORIGINAIS
# ============================================================

print(
    "\nCarregando as bases originais..."
)

train = pd.read_csv(
    arquivo_train
)

test = pd.read_csv(
    arquivo_test
)


# ============================================================
# 07. VERIFICAR DIMENSÕES DAS BASES ORIGINAIS
# ============================================================

print(
    "\nDIMENSÕES DAS BASES ORIGINAIS"
)

print(
    "=" * 100
)

print(
    "fraudTrain:",
    train.shape
)

print(
    "fraudTest: ",
    test.shape
)


# ============================================================
# 08. VERIFICAR SE AS FEATURES SÃO IGUAIS
# ============================================================

if list(train.columns) == list(test.columns):

    print(
        "\nAs duas bases possuem "
        "exatamente as mesmas features."
    )

else:

    print(
        "\nATENÇÃO:"
    )

    print(
        "As bases possuem diferenças "
        "nas features."
    )

    print(
        "\nFeatures somente no Train:"
    )

    print(
        set(train.columns)
        - set(test.columns)
    )

    print(
        "\nFeatures somente no Test:"
    )

    print(
        set(test.columns)
        - set(train.columns)
    )

    raise ValueError(
        "Train e Test possuem estruturas diferentes."
    )


# ============================================================
# 09. REMOVER O IDENTIFICADOR ANTIGO
# ============================================================

if "Unnamed: 0" in train.columns:

    train = train.drop(
        columns=["Unnamed: 0"]
    )

if "Unnamed: 0" in test.columns:

    test = test.drop(
        columns=["Unnamed: 0"]
    )


# ============================================================
# 10. JUNTAR TRAIN E TEST
# ============================================================

dataset_primario = pd.concat(
    [
        train,
        test
    ],
    axis=0,
    ignore_index=True
)


# ============================================================
# 11. CRIAR NOVO IDENTIFICADOR ÚNICO
# ============================================================

dataset_primario.insert(
    0,
    "NID",
    range(
        1,
        len(dataset_primario) + 1
    )
)


# ============================================================
# 12. VERIFICAR O NOVO IDENTIFICADOR
# ============================================================

print(
    "\nVERIFICAÇÃO DO NID"
)

print(
    "=" * 100
)

print(
    "Primeiro NID:",
    dataset_primario["NID"].min()
)

print(
    "Último NID:",
    dataset_primario["NID"].max()
)

print(
    "Quantidade de NIDs únicos:",
    dataset_primario["NID"].nunique()
)

print(
    "Quantidade de NIDs duplicados:",
    dataset_primario["NID"]
    .duplicated()
    .sum()
)


# ============================================================
# 13. VERIFICAR DIMENSÃO DO DATASET PRIMÁRIO
# ============================================================

print(
    "\nDIMENSÃO DO DATASET PRIMÁRIO"
)

print(
    "=" * 100
)

print(
    "Linhas:",
    dataset_primario.shape[0]
)

print(
    "Features:",
    dataset_primario.shape[1]
)


# ============================================================
# 14. SALVAR O DATASET PRIMÁRIO
# ============================================================

dataset_primario.to_csv(
    arquivo_dataset_primario,
    index=False
)


# ============================================================
# 15. CONFIRMAR SALVAMENTO
# ============================================================

if arquivo_dataset_primario.exists():

    print(
        "\nArquivo 'dataset_primario.csv' "
        "criado com sucesso!"
    )

    print(
        "Local:",
        arquivo_dataset_primario
    )

else:

    raise FileNotFoundError(
        "O dataset_primario.csv não foi criado."
    )

fraudTrain.csv já existe.
Download não necessário.

fraudTest.csv já existe.
Download não necessário.

Carregando as bases originais...

DIMENSÕES DAS BASES ORIGINAIS
fraudTrain: (1296675, 23)
fraudTest:  (555719, 23)

As duas bases possuem exatamente as mesmas features.

VERIFICAÇÃO DO NID
Primeiro NID: 1
Último NID: 1852394
Quantidade de NIDs únicos: 1852394
Quantidade de NIDs duplicados: 0

DIMENSÃO DO DATASET PRIMÁRIO
Linhas: 1852394
Features: 23

Arquivo 'dataset_primario.csv' criado com sucesso!
Local: /projeto_tcc_2026/dados/dados_primarios/dataset_primario.csv


## <span style="color:blue">TRANSFORMAÇÃO DO DATASET PRIMÁRIO EM DATASET FINAL</span> ## 

In [11]:
# ============================================================
# 01. CARREGAR O DATASET PRIMÁRIO
# ============================================================

df = pd.read_csv("/projeto_tcc_2026/dados/dados_primarios/dataset_primario.csv")

print("Dimensão inicial:", df.shape)


# ============================================================
# 02. VERIFICAR O IDENTIFICADOR NID
# ============================================================

print("\nVERIFICAÇÃO INICIAL DO NID")
print("=" * 100)

print("Primeiro NID:", df["NID"].min())
print("Último NID:", df["NID"].max())
print("NIDs únicos:", df["NID"].nunique())
print("NIDs duplicados:", df["NID"].duplicated().sum())


# ============================================================
# 03. REMOVER FEATURES NÃO UTILIZADAS
# ============================================================

colunas_remover = [
    "city",
    "state",
    "zip",
    "trans_num",
    "unix_time",
    "street"
]

df = df.drop(
    columns=[
        coluna
        for coluna in colunas_remover
        if coluna in df.columns
    ]
)


# ============================================================
# 04. RENOMEAR TARGET
# ============================================================

if "is_fraud" in df.columns:
    df = df.rename(
        columns={"is_fraud": "TARGET"}
    )

elif "is_fraude" in df.columns:
    df = df.rename(
        columns={"is_fraude": "TARGET"}
    )


# ============================================================
# 05. TRATAR DATA E HORA DA TRANSAÇÃO
# ============================================================

df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


# ------------------------------------------------------------
# DIA DO MÊS
# ------------------------------------------------------------

df["TRANS_DAY"] = (
    df["trans_date_trans_time"]
    .dt.day
)


# ------------------------------------------------------------
# DIA DA SEMANA
# ------------------------------------------------------------

mapa_dias = {
    0: "Segunda",
    1: "Terca",
    2: "Quarta",
    3: "Quinta",
    4: "Sexta",
    5: "Sabado",
    6: "Domingo"
}

df["TRANS_WEEK"] = (
    df["trans_date_trans_time"]
    .dt.dayofweek
    .map(mapa_dias)
)


# ------------------------------------------------------------
# ANO
# ------------------------------------------------------------

df["TRANS_YEAR"] = (
    df["trans_date_trans_time"]
    .dt.year
)


# ------------------------------------------------------------
# MÊS EM SENO E COSSENO
# ------------------------------------------------------------

mes = df["trans_date_trans_time"].dt.month

df["TRANS_MONTH_SEN"] = np.sin(
    2 * np.pi * (mes - 1) / 12
)

df["TRANS_MONTH_COS"] = np.cos(
    2 * np.pi * (mes - 1) / 12
)


# ------------------------------------------------------------
# HORÁRIO EM SENO E COSSENO
# Considerando hora + minuto + segundo
# ------------------------------------------------------------

hora_decimal = (
    df["trans_date_trans_time"].dt.hour
    + df["trans_date_trans_time"].dt.minute / 60
    + df["trans_date_trans_time"].dt.second / 3600
)

df["TRANS_HOUR_SEN"] = np.sin(
    2 * np.pi * hora_decimal / 24
)

df["TRANS_HOUR_COS"] = np.cos(
    2 * np.pi * hora_decimal / 24
)


# ------------------------------------------------------------
# REMOVER DATA/HORA ORIGINAL
# ------------------------------------------------------------

df = df.drop(
    columns=["trans_date_trans_time"]
)


# ============================================================
# 06. RENOMEAR FEATURES DA TRANSAÇÃO E DO REMETENTE
# ============================================================

df = df.rename(
    columns={
        "amt": "TRANS_VALUE",
        "cc_num": "TRANS_NUM_CARD",

        "job": "SEND_JOB",
        "gender": "SEND_GENDER",

        "lat": "SEND_LAT_REGISTER",
        "long": "SEND_LONG_REGISTER",
        "city_pop": "SEND_POP_REGISTER"
    }
)


# ============================================================
# 07. CRIAR NOME COMPLETO DO REMETENTE
# ============================================================

df["SEND_NAME"] = (
    df["first"]
    .fillna("")
    .astype(str)
    .str.strip()

    + " "

    + df["last"]
    .fillna("")
    .astype(str)
    .str.strip()
).str.strip()


# ============================================================
# 08. TRANSFORMAR DATA DE NASCIMENTO EM IDADE
# ============================================================

df["dob"] = pd.to_datetime(
    df["dob"],
    errors="coerce"
)


# ------------------------------------------------------------
# DATA DE REFERÊNCIA FIXA
# ------------------------------------------------------------

data_referencia = pd.Timestamp("2026-09-03")


# ------------------------------------------------------------
# IDADE EM ANOS DECIMAIS
# ------------------------------------------------------------

df["SEND_AGE"] = (
    (data_referencia - df["dob"]).dt.total_seconds()
    / (365.2425 * 24 * 60 * 60)
).round(2)


# ------------------------------------------------------------
# REMOVER FEATURES ORIGINAIS
# ------------------------------------------------------------

df = df.drop(
    columns=[
        "first",
        "last",
        "dob"
    ]
)


# ============================================================
# 09. RENOMEAR FEATURES DO RECEBEDOR
# ============================================================

df = df.rename(
    columns={
        "merchant": "RECIVE_LOC",
        "category": "RECIVE_CATEGORY",
        "merch_lat": "RECIVE_LAT",
        "merch_long": "RECIVE_LONG"
    }
)


# ============================================================
# 10. ADICIONAR NOMENCLATURA DOS FUTUROS ENCODERS
# ============================================================

df = df.rename(
    columns={

        # ----------------------------------------------------
        # IDENTIFICADOR
        # ----------------------------------------------------

        "NID": "NID_ALPHA",

        # ----------------------------------------------------
        # FUTURO FREQUENCY ENCODING
        # ----------------------------------------------------

        "TRANS_NUM_CARD": "TRANS_NUM_CARD_FEWF",
        "RECIVE_LOC": "RECIVE_LOC_FEWF",
        "SEND_JOB": "SEND_JOB_FEWF",
        "SEND_NAME": "SEND_NAME_FEWF",

        # ----------------------------------------------------
        # FUTURO BINARY ENCODING
        # ----------------------------------------------------

        "SEND_GENDER": "SEND_GENDER_BE",
        "TRANS_YEAR": "TRANS_YEAR_BE",

        # ----------------------------------------------------
        # FUTURO ONE-HOT ENCODING
        # COM TRATAMENTO DE CATEGORIAS DESCONHECIDAS
        # ----------------------------------------------------

        "RECIVE_CATEGORY": "RECIVE_CATEGORY_OHEWI",
        "TRANS_WEEK": "TRANS_WEEK_OHEWI",

        # ----------------------------------------------------
        # VARIÁVEL RESPOSTA
        # ----------------------------------------------------

        "TARGET": "TARGET_OMEGA"
    }
)


# ============================================================
# 11. ORGANIZAR NID PRIMEIRO E TARGET POR ÚLTIMO
# ============================================================

colunas_meio = [
    coluna
    for coluna in df.columns
    if coluna not in [
        "NID_ALPHA",
        "TARGET_OMEGA"
    ]
]

df = df[
    ["NID_ALPHA"]
    + colunas_meio
    + ["TARGET_OMEGA"]
]


# ============================================================
# 12. VERIFICAÇÕES DO DATASET FINAL
# ============================================================

print("\n" + "=" * 100)
print("DATASET FINAL")
print("=" * 100)

print(f"Linhas: {df.shape[0]}")
print(f"Features: {df.shape[1]}")


# ============================================================
# 13. MOSTRAR FEATURES FINAIS
# ============================================================

print("\nFEATURES FINAIS")
print("=" * 100)

for i, coluna in enumerate(
    df.columns,
    start=1
):
    print(f"{i}. {coluna}")


# ============================================================
# 14. VERIFICAR SEND_AGE
# ============================================================

print("\nVERIFICAÇÃO DE SEND_AGE")
print("=" * 100)

print(
    "Data de referência utilizada:",
    data_referencia.strftime("%d/%m/%Y")
)

print(
    "Menor idade:",
    df["SEND_AGE"].min()
)

print(
    "Maior idade:",
    df["SEND_AGE"].max()
)


# ============================================================
# 15. VERIFICAR TARGET
# ============================================================

print("\nVERIFICAÇÃO DO TARGET")
print("=" * 100)

print(
    df["TARGET_OMEGA"]
    .value_counts()
    .sort_index()
)


# ============================================================
# 16. SALVAR DATASET FINAL
# ============================================================


df.to_csv(
    "/projeto_tcc_2026/dados/dataset_final/bruto/dataset_finalbruto.csv",
    index=False
)

Dimensão inicial: (1852394, 23)

VERIFICAÇÃO INICIAL DO NID
Primeiro NID: 1
Último NID: 1852394
NIDs únicos: 1852394
NIDs duplicados: 0

DATASET FINAL
Linhas: 1852394
Features: 22

FEATURES FINAIS
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. RECIVE_LOC_FEWF
4. RECIVE_CATEGORY_OHEWI
5. TRANS_VALUE
6. SEND_GENDER_BE
7. SEND_LAT_REGISTER
8. SEND_LONG_REGISTER
9. SEND_POP_REGISTER
10. SEND_JOB_FEWF
11. RECIVE_LAT
12. RECIVE_LONG
13. TRANS_DAY
14. TRANS_WEEK_OHEWI
15. TRANS_YEAR_BE
16. TRANS_MONTH_SEN
17. TRANS_MONTH_COS
18. TRANS_HOUR_SEN
19. TRANS_HOUR_COS
20. SEND_NAME_FEWF
21. SEND_AGE
22. TARGET_OMEGA

VERIFICAÇÃO DE SEND_AGE
Data de referência utilizada: 03/09/2026
Menor idade: 21.59
Maior idade: 101.84

VERIFICAÇÃO DO TARGET
TARGET_OMEGA
0    1842743
1       9651
Name: count, dtype: int64


## <span style="color:blue">TRANSFORMACAO DE DATASET EM CSV PARA PARQUET</span> ## 

In [14]:

# ============================================================
# 01. CAMINHOS
# ============================================================

arquivo_csv = Path(
    "/projeto_tcc_2026/"
    "dados/dataset_final/bruto/"
    "dataset_finalbruto.csv"
)

arquivo_parquet = Path(
    "/projeto_tcc_2026/"
    "dados/dataset_final/otimizado/otimizacao_inicial/"
    "dataset_final_otimizado.parquet"
)


# ============================================================
# 02. VERIFICAR SE O CSV EXISTE
# ============================================================

if not arquivo_csv.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo_csv}"
    )


# ============================================================
# 03. ABRIR CSV EM MODO DE LEITURA POR BLOCOS
# ============================================================

reader = pacsv.open_csv(
    arquivo_csv,
    read_options=pacsv.ReadOptions(
        block_size=64 * 1024 * 1024
    )
)


# ============================================================
# 04. CONVERTER CSV → PARQUET
# ============================================================

writer = None

try:

    for i, batch in enumerate(reader, start=1):

        if writer is None:

            writer = pq.ParquetWriter(
                arquivo_parquet,
                batch.schema,
                compression="zstd",
                use_dictionary=True
            )

        writer.write_batch(batch)

        print(
            f"Bloco {i} convertido "
            f"({batch.num_rows:,} linhas)"
        )

finally:

    if writer is not None:
        writer.close()


# ============================================================
# 6. CONFIRMAR CRIAÇÃO
# ============================================================

if arquivo_parquet.exists():

    tamanho_csv = arquivo_csv.stat().st_size / 1024**2
    tamanho_parquet = arquivo_parquet.stat().st_size / 1024**2

    reducao = (
        1 - tamanho_parquet / tamanho_csv
    ) * 100

    print("\n" + "=" * 100)
    print("CONVERSÃO CONCLUÍDA")
    print("=" * 100)

    print(
        f"CSV:     {tamanho_csv:.2f} MB"
    )

    print(
        f"Parquet: {tamanho_parquet:.2f} MB"
    )

    print(
        f"Redução: {reducao:.2f}%"
    )

    print(
        "\nArquivo criado em:"
    )

    print(
        arquivo_parquet
    )

Bloco 1 convertido (279,819 linhas)
Bloco 2 convertido (272,075 linhas)
Bloco 3 convertido (271,340 linhas)
Bloco 4 convertido (275,201 linhas)
Bloco 5 convertido (273,034 linhas)
Bloco 6 convertido (271,830 linhas)
Bloco 7 convertido (209,095 linhas)

CONVERSÃO CONCLUÍDA
CSV:     433.95 MB
Parquet: 70.54 MB
Redução: 83.74%

Arquivo criado em:
/projeto_tcc_2026/dados/dataset_final/otimizado/otimizacao_inicial/dataset_final_otimizado.parquet


## <span style="color:blue">OTIMIZACAO FINAL DO DATASET </span> ## 

In [17]:

# ============================================================
# 01. CAMINHOS
# ============================================================

arquivo_entrada = Path(
    "/projeto_tcc_2026/"
    "dados/dataset_final/otimizado/"
    "otimizacao_inicial/"
    "dataset_final_otimizado.parquet"
)

pasta_saida = Path(
    "/projeto_tcc_2026/"
    "dados/dataset_final/otimizado/"
    "otimizacao_final"
)

arquivo_saida = (
    pasta_saida /
    "dataset_prime.parquet"
)


# ============================================================
# 02. VERIFICAR SE O ARQUIVO EXISTE
# ============================================================

if not arquivo_entrada.exists():

    raise FileNotFoundError(
        f"Arquivo não encontrado:\n{arquivo_entrada}"
    )


# ============================================================
# 03. GARANTIR QUE A PASTA DE SAÍDA EXISTA
# ============================================================

pasta_saida.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 04. CARREGAR O PARQUET
# ============================================================

print("Carregando dataset...")

df = pd.read_parquet(
    arquivo_entrada,
    engine="pyarrow"
)

print("Dataset carregado com sucesso!")


# ============================================================
# 05. REGISTRAR INFORMAÇÕES ANTES DA OTIMIZAÇÃO
# ============================================================

tipos_antes = (
    df.dtypes
    .astype(str)
    .copy()
)

memoria_colunas_antes = (
    df.memory_usage(
        index=False,
        deep=True
    )
    .copy()
)

memoria_total_antes = (
    memoria_colunas_antes.sum()
)

tamanho_disco_antes = (
    arquivo_entrada.stat().st_size
)


# ============================================================
# 06. OTIMIZAÇÃO 1
# FLOATS
# float64 → float32 QUANDO POSSÍVEL
# ============================================================

print("\nOtimizando números decimais...")

colunas_float = (
    df.select_dtypes(
        include=["floating"]
    )
    .columns
)

for coluna in colunas_float:

    df[coluna] = pd.to_numeric(
        df[coluna],
        downcast="float"
    )


# ============================================================
# 07. OTIMIZAÇÃO 2
# INTEIROS
# int64 → int32 / int16 / int8
# ============================================================

print("Otimizando números inteiros...")

colunas_int = (
    df.select_dtypes(
        include=["integer"]
    )
    .columns
)

for coluna in colunas_int:

    df[coluna] = pd.to_numeric(
        df[coluna],
        downcast="integer"
    )


# ============================================================
# 08. OTIMIZAÇÃO 3
# STRING / OBJECT → CATEGORY
# ============================================================

print("Analisando variáveis categóricas...")

colunas_texto = (
    df.select_dtypes(
        include=[
            "object",
            "string"
        ]
    )
    .columns
)

colunas_convertidas_category = []

for coluna in colunas_texto:

    quantidade_unicos = (
        df[coluna]
        .nunique(
            dropna=False
        )
    )

    proporcao_unicos = (
        quantidade_unicos
        / len(df)
    )

    # --------------------------------------------------------
    # REGRA:
    # converte para category quando até 5% dos valores
    # da coluna são distintos.
    # --------------------------------------------------------

    if proporcao_unicos <= 0.05:

        df[coluna] = (
            df[coluna]
            .astype("category")
        )

        colunas_convertidas_category.append(
            coluna
        )


# ============================================================
# 09. REGISTRAR INFORMAÇÕES DEPOIS DA OTIMIZAÇÃO
# ============================================================

tipos_depois = (
    df.dtypes
    .astype(str)
    .copy()
)

memoria_colunas_depois = (
    df.memory_usage(
        index=False,
        deep=True
    )
    .copy()
)

memoria_total_depois = (
    memoria_colunas_depois.sum()
)


# ============================================================
# 10. CRIAR RESUMO POR FEATURE
# ============================================================

resumo_colunas = pd.DataFrame({

    "TIPO_ANTES":
        tipos_antes,

    "TIPO_DEPOIS":
        tipos_depois,

    "MEMORIA_ANTES_MB":
        memoria_colunas_antes
        / 1024**2,

    "MEMORIA_DEPOIS_MB":
        memoria_colunas_depois
        / 1024**2
})


# ============================================================
# 11. CALCULAR ECONOMIA POR FEATURE
# ============================================================

resumo_colunas[
    "ECONOMIA_MB"
] = (

    resumo_colunas[
        "MEMORIA_ANTES_MB"
    ]

    -

    resumo_colunas[
        "MEMORIA_DEPOIS_MB"
    ]
)


resumo_colunas[
    "REDUCAO_%"
] = (

    resumo_colunas[
        "ECONOMIA_MB"
    ]

    /

    resumo_colunas[
        "MEMORIA_ANTES_MB"
    ]

    * 100
)


resumo_colunas[
    "TIPO_MUDOU"
] = (

    resumo_colunas[
        "TIPO_ANTES"
    ]

    !=

    resumo_colunas[
        "TIPO_DEPOIS"
    ]
)


# ============================================================
# 12. SALVAR NOVO PARQUET
# ============================================================

print("\nSalvando novo Parquet...")

df.to_parquet(
    arquivo_saida,
    engine="pyarrow",
    compression="zstd",
    index=False
)

print("Arquivo salvo com sucesso!")


# ============================================================
# 13. TAMANHO DO NOVO ARQUIVO EM DISCO
# ============================================================

tamanho_disco_depois = (
    arquivo_saida.stat().st_size
)


# ============================================================
# 14. CALCULAR REDUÇÃO DE RAM
# ============================================================

reducao_memoria = (

    1
    - (
        memoria_total_depois
        /
        memoria_total_antes
    )

) * 100


# ============================================================
# 15. CALCULAR REDUÇÃO EM DISCO
# ============================================================

reducao_disco = (

    1
    - (
        tamanho_disco_depois
        /
        tamanho_disco_antes
    )

) * 100


# ============================================================
# 16. RESUMO GERAL DA BASE
# ============================================================

print("\n" + "=" * 100)

print("RESUMO DO DATASET OTIMIZADO")

print("=" * 100)


print(
    f"\nLinhas: "
    f"{df.shape[0]:,}"
)

print(
    f"Features: "
    f"{df.shape[1]}"
)

print(
    f"Valores ausentes: "
    f"{df.isna().sum().sum():,}"
)


# ============================================================
# 17. MEMÓRIA RAM
# ============================================================

print("\n" + "-" * 100)

print("MEMÓRIA RAM")

print("-" * 100)


print(
    f"Antes: "
    f"{memoria_total_antes / 1024**2:.2f} MB"
)

print(
    f"Depois: "
    f"{memoria_total_depois / 1024**2:.2f} MB"
)

print(
    f"Economia: "
    f"{(memoria_total_antes - memoria_total_depois) / 1024**2:.2f} MB"
)

print(
    f"Redução: "
    f"{reducao_memoria:.2f}%"
)


# ============================================================
# 18. ESPAÇO EM DISCO
# ============================================================

print("\n" + "-" * 100)

print("ESPAÇO EM DISCO")

print("-" * 100)


print(
    f"Arquivo anterior: "
    f"{tamanho_disco_antes / 1024**2:.2f} MB"
)

print(
    f"Arquivo novo: "
    f"{tamanho_disco_depois / 1024**2:.2f} MB"
)

print(
    f"Economia: "
    f"{(tamanho_disco_antes - tamanho_disco_depois) / 1024**2:.2f} MB"
)

print(
    f"Redução: "
    f"{reducao_disco:.2f}%"
)


# ============================================================
# 19. QUANTIDADE DE TIPOS
# ============================================================

print("\n" + "-" * 100)

print("TIPOS DE DADOS ANTES")

print("-" * 100)

display(
    tipos_antes
    .value_counts()
    .rename("QUANTIDADE")
    .to_frame()
)


print("\n" + "-" * 100)

print("TIPOS DE DADOS DEPOIS")

print("-" * 100)

display(
    tipos_depois
    .value_counts()
    .rename("QUANTIDADE")
    .to_frame()
)


# ============================================================
# 20. FEATURES QUE MUDARAM DE TIPO
# ============================================================

print("\n" + "-" * 100)

print("FEATURES QUE MUDARAM DE TIPO")

print("-" * 100)

display(

    resumo_colunas[
        resumo_colunas[
            "TIPO_MUDOU"
        ]
    ]

    .sort_values(
        "ECONOMIA_MB",
        ascending=False
    )

    .round(2)
)


# ============================================================
# 21. TODAS AS FEATURES
# ============================================================

print("\n" + "-" * 100)

print("RESUMO COMPLETO DAS FEATURES")

print("-" * 100)

display(

    resumo_colunas
    .sort_values(
        "MEMORIA_ANTES_MB",
        ascending=False
    )
    .round(2)
)


# ============================================================
# 22. COLUNAS CONVERTIDAS PARA CATEGORY
# ============================================================

print("\n" + "-" * 100)

print("FEATURES CONVERTIDAS PARA CATEGORY")

print("-" * 100)

if colunas_convertidas_category:

    for coluna in colunas_convertidas_category:

        print(
            f"- {coluna}"
        )

else:

    print(
        "Nenhuma feature foi convertida para category."
    )


# ============================================================
# 23. LOCAL DO NOVO DATASET
# ============================================================

print("\n" + "=" * 100)

print("ARQUIVO FINAL")

print("=" * 100)

print(
    arquivo_saida
)

Carregando dataset...
Dataset carregado com sucesso!

Otimizando números decimais...
Otimizando números inteiros...
Analisando variáveis categóricas...

Salvando novo Parquet...
Arquivo salvo com sucesso!

RESUMO DO DATASET OTIMIZADO

Linhas: 1,852,394
Features: 22
Valores ausentes: 0

----------------------------------------------------------------------------------------------------
MEMÓRIA RAM
----------------------------------------------------------------------------------------------------
Antes: 442.01 MB
Depois: 129.02 MB
Economia: 312.99 MB
Redução: 70.81%

----------------------------------------------------------------------------------------------------
ESPAÇO EM DISCO
----------------------------------------------------------------------------------------------------
Arquivo anterior: 70.54 MB
Arquivo novo: 49.73 MB
Economia: 20.81 MB
Redução: 29.50%

----------------------------------------------------------------------------------------------------
TIPOS DE DADOS ANTES
-

,QUANTIDADE
float64,10
int64,6
str,6



----------------------------------------------------------------------------------------------------
TIPOS DE DADOS DEPOIS
----------------------------------------------------------------------------------------------------


,QUANTIDADE
float32,9
category,6
int32,2
int8,2
int64,1
float64,1
int16,1



----------------------------------------------------------------------------------------------------
FEATURES QUE MUDARAM DE TIPO
----------------------------------------------------------------------------------------------------


,TIPO_ANTES,TIPO_DEPOIS,MEMORIA_ANTES_MB,MEMORIA_DEPOIS_MB,ECONOMIA_MB,REDUCAO_%,TIPO_MUDOU
RECIVE_LOC_FEWF,str,category,54.99,3.55,51.44,93.54,True
SEND_JOB_FEWF,str,category,49.87,3.55,46.33,92.89,True
SEND_NAME_FEWF,str,category,37.44,3.55,33.89,90.51,True
RECIVE_CATEGORY_OHEWI,str,category,32.73,1.77,30.96,94.60,True
TRANS_WEEK_OHEWI,str,category,24.95,1.77,23.18,92.92,True
SEND_GENDER_BE,str,category,15.90,1.77,14.13,88.89,True
TARGET_OMEGA,int64,int8,14.13,1.77,12.37,87.50,True
TRANS_DAY,int64,int8,14.13,1.77,12.37,87.50,True
TRANS_YEAR_BE,int64,int16,14.13,3.53,10.60,75.00,True
NID_ALPHA,int64,int32,14.13,7.07,7.07,50.00,True



----------------------------------------------------------------------------------------------------
RESUMO COMPLETO DAS FEATURES
----------------------------------------------------------------------------------------------------


,TIPO_ANTES,TIPO_DEPOIS,MEMORIA_ANTES_MB,MEMORIA_DEPOIS_MB,ECONOMIA_MB,REDUCAO_%,TIPO_MUDOU
RECIVE_LOC_FEWF,str,category,54.99,3.55,51.44,93.54,True
SEND_JOB_FEWF,str,category,49.87,3.55,46.33,92.89,True
SEND_NAME_FEWF,str,category,37.44,3.55,33.89,90.51,True
RECIVE_CATEGORY_OHEWI,str,category,32.73,1.77,30.96,94.60,True
TRANS_WEEK_OHEWI,str,category,24.95,1.77,23.18,92.92,True
SEND_GENDER_BE,str,category,15.90,1.77,14.13,88.89,True
TRANS_VALUE,float64,float64,14.13,14.13,0.00,0.00,False
TRANS_NUM_CARD_FEWF,int64,int64,14.13,14.13,0.00,0.00,False
NID_ALPHA,int64,int32,14.13,7.07,7.07,50.00,True
SEND_LONG_REGISTER,float64,float32,14.13,7.07,7.07,50.00,True



----------------------------------------------------------------------------------------------------
FEATURES CONVERTIDAS PARA CATEGORY
----------------------------------------------------------------------------------------------------
- RECIVE_LOC_FEWF
- RECIVE_CATEGORY_OHEWI
- SEND_GENDER_BE
- SEND_JOB_FEWF
- TRANS_WEEK_OHEWI
- SEND_NAME_FEWF

ARQUIVO FINAL
/projeto_tcc_2026/dados/dataset_final/otimizado/otimizacao_final/dataset_prime.parquet


## <span style="color:blue">COMPARACAO DE DATASET PRIMARIO COM O PRIME</span> ## 

In [21]:
# ============================================================
# 01. CAMINHOS DOS ARQUIVOS
# ============================================================

arquivo_csv = Path(
    "/projeto_tcc_2026/"
    "dados/dados_primarios/"
    "dataset_primario.csv"
)

arquivo_parquet = Path(
    "/projeto_tcc_2026/"
    "dados/dataset_final/otimizado/"
    "otimizacao_final/"
    "dataset_prime.parquet"
)


# ============================================================
# 02. VERIFICAR SE OS ARQUIVOS EXISTEM
# ============================================================

if not arquivo_csv.exists():
    raise FileNotFoundError(
        f"CSV não encontrado:\n{arquivo_csv}"
    )

if not arquivo_parquet.exists():
    raise FileNotFoundError(
        f"Parquet não encontrado:\n{arquivo_parquet}"
    )


# ============================================================
# 03. TAMANHO DOS ARQUIVOS EM DISCO
# ============================================================

tamanho_csv_mb = (
    arquivo_csv.stat().st_size
    / 1024**2
)

tamanho_parquet_mb = (
    arquivo_parquet.stat().st_size
    / 1024**2
)

reducao_disco = (
    1
    - tamanho_parquet_mb
    / tamanho_csv_mb
) * 100


# ============================================================
# 04. CARREGAR DATASET PRIMÁRIO CSV
# ============================================================

print("Carregando dataset_primario.csv...")

df_csv = pd.read_csv(
    arquivo_csv
)

print("CSV carregado!")


# ============================================================
# 05. RESUMO DO DATASET PRIMÁRIO
# ============================================================

resumo_csv = {
    "ARQUIVO": "dataset_primario.csv",
    "LINHAS": df_csv.shape[0],
    "FEATURES": df_csv.shape[1],
    "VALORES_AUSENTES": int(
        df_csv.isna().sum().sum()
    ),
    "LINHAS_DUPLICADAS": int(
        df_csv.duplicated().sum()
    ),
    "MEMORIA_RAM_MB": (
        df_csv.memory_usage(
            deep=True
        ).sum()
        / 1024**2
    ),
    "TAMANHO_DISCO_MB": tamanho_csv_mb
}

colunas_csv = list(
    df_csv.columns
)

tipos_csv = (
    df_csv.dtypes
    .astype(str)
    .to_dict()
)


# ============================================================
# 06. VERIFICAÇÕES IMPORTANTES DO CSV
# ============================================================

nid_csv = None
target_csv = None

if "NID" in df_csv.columns:

    nid_csv = {
        "MINIMO": df_csv["NID"].min(),
        "MAXIMO": df_csv["NID"].max(),
        "UNICOS": df_csv["NID"].nunique(),
        "DUPLICADOS": (
            df_csv["NID"]
            .duplicated()
            .sum()
        )
    }


if "is_fraud" in df_csv.columns:

    target_csv = (
        df_csv["is_fraud"]
        .value_counts()
        .sort_index()
        .to_dict()
    )


# ============================================================
# 07. LIBERAR O CSV DA MEMÓRIA
# ============================================================

del df_csv
gc.collect()

print(
    "Dataset CSV removido da memória."
)


# ============================================================
# 08. CARREGAR DATASET PARQUET
# ============================================================

print(
    "\nCarregando dataset_prime.parquet..."
)

df_parquet = pd.read_parquet(
    arquivo_parquet,
    engine="pyarrow"
)

print("Parquet carregado!")


# ============================================================
# 09. RESUMO DO DATASET PARQUET
# ============================================================

resumo_parquet = {
    "ARQUIVO": "dataset_prime.parquet",
    "LINHAS": df_parquet.shape[0],
    "FEATURES": df_parquet.shape[1],
    "VALORES_AUSENTES": int(
        df_parquet.isna().sum().sum()
    ),
    "LINHAS_DUPLICADAS": int(
        df_parquet.duplicated().sum()
    ),
    "MEMORIA_RAM_MB": (
        df_parquet.memory_usage(
            deep=True
        ).sum()
        / 1024**2
    ),
    "TAMANHO_DISCO_MB": tamanho_parquet_mb
}

colunas_parquet = list(
    df_parquet.columns
)

tipos_parquet = (
    df_parquet.dtypes
    .astype(str)
    .to_dict()
)


# ============================================================
# 10. VERIFICAÇÕES IMPORTANTES DO PARQUET
# ============================================================

nid_parquet = None
target_parquet = None

if "NID_ALPHA" in df_parquet.columns:

    nid_parquet = {
        "MINIMO": (
            df_parquet["NID_ALPHA"].min()
        ),
        "MAXIMO": (
            df_parquet["NID_ALPHA"].max()
        ),
        "UNICOS": (
            df_parquet["NID_ALPHA"].nunique()
        ),
        "DUPLICADOS": (
            df_parquet["NID_ALPHA"]
            .duplicated()
            .sum()
        )
    }


if "TARGET_OMEGA" in df_parquet.columns:

    target_parquet = (
        df_parquet["TARGET_OMEGA"]
        .value_counts()
        .sort_index()
        .to_dict()
    )


# ============================================================
# 11. COMPARAÇÃO GERAL
# ============================================================

comparacao_geral = pd.DataFrame(
    [
        resumo_csv,
        resumo_parquet
    ]
)

print(
    "\n" + "=" * 100
)

print(
    "COMPARAÇÃO GERAL DOS DATASETS"
)

print(
    "=" * 100
)

display(
    comparacao_geral.round(2)
)


# ============================================================
# 12. ECONOMIA DE ESPAÇO EM DISCO
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "ECONOMIA DE ESPAÇO EM DISCO"
)

print(
    "=" * 100
)

print(
    f"CSV:     "
    f"{tamanho_csv_mb:.2f} MB"
)

print(
    f"Parquet: "
    f"{tamanho_parquet_mb:.2f} MB"
)

print(
    f"Economia: "
    f"{tamanho_csv_mb - tamanho_parquet_mb:.2f} MB"
)

print(
    f"Redução: "
    f"{reducao_disco:.2f}%"
)


# ============================================================
# 13. COMPARAR QUANTIDADE DE LINHAS
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "COMPARAÇÃO DAS LINHAS"
)

print(
    "=" * 100
)

if resumo_csv["LINHAS"] == resumo_parquet["LINHAS"]:

    print(
        "OK - Os dois datasets possuem "
        "a mesma quantidade de linhas."
    )

else:

    print(
        "ATENÇÃO - A quantidade de linhas "
        "é diferente."
    )

    print(
        "CSV:",
        resumo_csv["LINHAS"]
    )

    print(
        "Parquet:",
        resumo_parquet["LINHAS"]
    )


# ============================================================
# 14. COMPARAR NID
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "COMPARAÇÃO DO IDENTIFICADOR"
)

print(
    "=" * 100
)

if (
    nid_csv is not None
    and nid_parquet is not None
):

    comparacao_nid = pd.DataFrame(
        {
            "DATASET_PRIMARIO": nid_csv,
            "DATASET_FINAL": nid_parquet
        }
    )

    display(
        comparacao_nid
    )


# ============================================================
# 15. COMPARAR TARGET
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "COMPARAÇÃO DO TARGET"
)

print(
    "=" * 100
)

if (
    target_csv is not None
    and target_parquet is not None
):

    comparacao_target = pd.DataFrame(
        {
            "DATASET_PRIMARIO":
                pd.Series(target_csv),

            "DATASET_FINAL":
                pd.Series(target_parquet)
        }
    )

    display(
        comparacao_target
    )


# ============================================================
# 16. FEATURES DO DATASET PRIMÁRIO
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "FEATURES DO DATASET PRIMÁRIO"
)

print(
    "=" * 100
)

for i, coluna in enumerate(
    colunas_csv,
    start=1
):
    print(
        f"{i}. {coluna}"
    )


# ============================================================
# 17. FEATURES DO DATASET FINAL
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "FEATURES DO DATASET FINAL"
)

print(
    "=" * 100
)

for i, coluna in enumerate(
    colunas_parquet,
    start=1
):
    print(
        f"{i}. {coluna}"
    )


# ============================================================
# 18. FEATURES REMOVIDAS
# ============================================================

features_removidas = sorted(
    set(colunas_csv)
    - set(colunas_parquet)
)

print(
    "\n" + "=" * 100
)

print(
    "FEATURES DO PRIMÁRIO QUE NÃO EXISTEM "
    "COM O MESMO NOME NO FINAL"
)

print(
    "=" * 100
)

for coluna in features_removidas:
    print(
        f"- {coluna}"
    )


# ============================================================
# 19. FEATURES NOVAS
# ============================================================

features_novas = sorted(
    set(colunas_parquet)
    - set(colunas_csv)
)

print(
    "\n" + "=" * 100
)

print(
    "FEATURES NOVAS OU RENOMEADAS "
    "NO DATASET FINAL"
)

print(
    "=" * 100
)

for coluna in features_novas:
    print(
        f"- {coluna}"
    )


# ============================================================
# 20. TIPOS DO DATASET PRIMÁRIO
# ============================================================

tabela_tipos_csv = pd.DataFrame(
    {
        "FEATURE":
            list(tipos_csv.keys()),

        "TIPO":
            list(tipos_csv.values())
    }
)


print(
    "\n" + "=" * 100
)

print(
    "TIPOS DE DADOS - DATASET PRIMÁRIO"
)

print(
    "=" * 100
)

display(
    tabela_tipos_csv
)


# ============================================================
# 21. TIPOS DO DATASET FINAL
# ============================================================

tabela_tipos_parquet = pd.DataFrame(
    {
        "FEATURE":
            list(tipos_parquet.keys()),

        "TIPO":
            list(tipos_parquet.values())
    }
)


print(
    "\n" + "=" * 100
)

print(
    "TIPOS DE DADOS - DATASET FINAL"
)

print(
    "=" * 100
)

display(
    tabela_tipos_parquet
)


# ============================================================
# 22. RESUMO FINAL
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "RESUMO FINAL"
)

print(
    "=" * 100
)

print(
    f"Linhas do primário: "
    f"{resumo_csv['LINHAS']:,}"
)

print(
    f"Linhas do final:    "
    f"{resumo_parquet['LINHAS']:,}"
)

print(
    f"\nFeatures do primário: "
    f"{resumo_csv['FEATURES']}"
)

print(
    f"Features do final:    "
    f"{resumo_parquet['FEATURES']}"
)

print(
    f"\nCSV em disco: "
    f"{tamanho_csv_mb:.2f} MB"
)

print(
    f"Parquet em disco: "
    f"{tamanho_parquet_mb:.2f} MB"
)

print(
    f"Redução em disco: "
    f"{reducao_disco:.2f}%"
)

Carregando dataset_primario.csv...
CSV carregado!
Dataset CSV removido da memória.

Carregando dataset_prime.parquet...
Parquet carregado!

COMPARAÇÃO GERAL DOS DATASETS


,ARQUIVO,LINHAS,FEATURES,VALORES_AUSENTES,LINHAS_DUPLICADAS,MEMORIA_RAM_MB,TAMANHO_DISCO_MB
0,dataset_primario.csv,1852394,23,0,0,609.41,472.78
1,dataset_prime.parquet,1852394,22,0,0,129.02,49.73



ECONOMIA DE ESPAÇO EM DISCO
CSV:     472.78 MB
Parquet: 49.73 MB
Economia: 423.06 MB
Redução: 89.48%

COMPARAÇÃO DAS LINHAS
OK - Os dois datasets possuem a mesma quantidade de linhas.

COMPARAÇÃO DO IDENTIFICADOR


,DATASET_PRIMARIO,DATASET_FINAL
MINIMO,1,1
MAXIMO,1852394,1852394
UNICOS,1852394,1852394
DUPLICADOS,0,0



COMPARAÇÃO DO TARGET


...


FEATURES DO DATASET PRIMÁRIO
1. NID
2. trans_date_trans_time
3. cc_num
4. merchant
5. category
6. amt
7. first
8. last
9. gender
10. street
11. city
12. state
13. zip
14. lat
15. long
16. city_pop
17. job
18. dob
19. trans_num
20. unix_time
21. merch_lat
22. merch_long
23. is_fraud

FEATURES DO DATASET FINAL
1. NID_ALPHA
2. TRANS_NUM_CARD_FEWF
3. RECIVE_LOC_FEWF
4. RECIVE_CATEGORY_OHEWI
5. TRANS_VALUE
6. SEND_GENDER_BE
7. SEND_LAT_REGISTER
8. SEND_LONG_REGISTER
9. SEND_POP_REGISTER
10. SEND_JOB_FEWF
11. RECIVE_LAT
12. RECIVE_LONG
13. TRANS_DAY
14. TRANS_WEEK_OHEWI
15. TRANS_YEAR_BE
16. TRANS_MONTH_SEN
17. TRANS_MONTH_COS
18. TRANS_HOUR_SEN
19. TRANS_HOUR_COS
20. SEND_NAME_FEWF
21. SEND_AGE
22. TARGET_OMEGA

FEATURES DO PRIMÁRIO QUE NÃO EXISTEM COM O MESMO NOME NO FINAL
- NID
- amt
- category
- cc_num
- city
- city_pop
- dob
- first
- gender
- is_fraud
- job
- last
- lat
- long
- merch_lat
- merch_long
- merchant
- state
- street
- trans_date_trans_time
- trans_num
- unix_time
- zip

F

,FEATURE,TIPO
0,NID,int64
1,trans_date_trans_time,str
2,cc_num,int64
3,merchant,str
4,category,str
5,amt,float64
6,first,str
7,last,str
8,gender,str
9,street,str



TIPOS DE DADOS - DATASET FINAL


,FEATURE,TIPO
0,NID_ALPHA,int32
1,TRANS_NUM_CARD_FEWF,int64
2,RECIVE_LOC_FEWF,category
3,RECIVE_CATEGORY_OHEWI,category
4,TRANS_VALUE,float64
5,SEND_GENDER_BE,category
6,SEND_LAT_REGISTER,float32
7,SEND_LONG_REGISTER,float32
8,SEND_POP_REGISTER,int32
9,SEND_JOB_FEWF,category



RESUMO FINAL
Linhas do primário: 1,852,394
Linhas do final:    1,852,394

Features do primário: 23
Features do final:    22

CSV em disco: 472.78 MB
Parquet em disco: 49.73 MB
Redução em disco: 89.48%


## <span style="color:blue">AMOSTRAGEM E TESTE DO DATASET PRIME</span> ## 

In [22]:
# ============================================================
# AMOSTRA ALEATÓRIA DE 10 LINHAS DO DATASET_PRIME
# ============================================================
arquivo_dataset_prime = Path(
    "/projeto_tcc_2026/"
    "dados/dataset_final/otimizado/"
    "otimizacao_final/"
    "dataset_prime.parquet"
)


df_prime = pd.read_parquet(
    arquivo_dataset_prime,
    engine="pyarrow"
)


amostra = df_prime.sample(
    n=10,
    random_state=42
)


display(amostra)

,NID_ALPHA,TRANS_NUM_CARD_FEWF,RECIVE_LOC_FEWF,RECIVE_CATEGORY_OHEWI,TRANS_VALUE,SEND_GENDER_BE,SEND_LAT_REGISTER,SEND_LONG_REGISTER,SEND_POP_REGISTER,SEND_JOB_FEWF,...,TRANS_DAY,TRANS_WEEK_OHEWI,TRANS_YEAR_BE,TRANS_MONTH_SEN,TRANS_MONTH_COS,TRANS_HOUR_SEN,TRANS_HOUR_COS,SEND_NAME_FEWF,SEND_AGE,TARGET_OMEGA
1541144,1541145,5359543825610251,"fraud_Jenkins, Hauck and Friesen",gas_transport,59.91,M,45.780102,-111.143898,18182,"Engineer, drilling",...,18,Sexta,2020,-8.660254e-01,-5.000000e-01,0.948807,-0.315856,Michael Francis,51.180000,0
1731581,1731582,5540636818935089,fraud_Jast-McDermott,shopping_pos,3.96,M,42.691101,-71.160500,76383,Geoscientist,...,5,Sabado,2020,-5.000000e-01,8.660254e-01,-0.998723,-0.050520,Kenneth Foster,41.419998,0
354659,354660,2720894374956739,fraud_Bartoletti-Wunsch,gas_transport,51.17,F,42.597801,-82.882301,16305,"Psychologist, sport and exercise",...,15,Sabado,2019,5.000000e-01,-8.660254e-01,0.153273,-0.988184,Audrey Hickman,99.279999,0
1493788,1493789,6011438889172900,"fraud_Roob, Conn and Tremblay",shopping_pos,2.06,F,34.285301,-91.333603,5161,Electrical engineer,...,29,Sabado,2020,-5.000000e-01,-8.660254e-01,-0.298971,0.954262,Allison Allen,33.410000,0
468148,468149,60495593109,"fraud_Kilback, Nitzsche and Leffler",travel,6.58,M,32.769901,-96.742996,1263321,Television camera operator,...,25,Quinta,2019,1.224647e-16,-1.000000e+00,-0.844756,-0.535151,Randall Dillon,83.779999,0
1389854,1389855,6011477612335392,"fraud_Cormier, Stracke and Thiel",entertainment,24.49,M,40.406200,-84.507599,2274,"Designer, television/film set",...,23,Quinta,2020,1.224647e-16,-1.000000e+00,-0.991340,-0.131319,Anthony Roberts,82.709999,0
1760906,1760907,2703186189652095,fraud_Skiles LLC,home,99.74,F,36.078800,-81.178101,3495,"Psychologist, counselling",...,11,Sexta,2020,-5.000000e-01,8.660254e-01,-0.229271,0.973363,Jennifer Banks,38.490002,0
469447,469448,4839615922685395,fraud_Vandervort-Funk,grocery_pos,147.12,M,39.013000,-86.545700,76,Social researcher,...,26,Sexta,2019,1.224647e-16,-1.000000e+00,0.962139,0.272560,Phillip Robertson,71.330002,0
1835018,1835019,3553629419254918,fraud_Sporer Inc,gas_transport,46.85,F,48.340000,-122.345596,85,"Research officer, political party",...,28,Segunda,2020,-5.000000e-01,8.660254e-01,0.911882,-0.410454,Sharon Johnson,42.000000,0
240727,240728,4481131401752,"fraud_Wintheiser, Dietrich and Schimmel",misc_pos,33.56,M,42.284801,-71.720497,35299,English as a second language teacher,...,30,Terca,2019,1.000000e+00,6.123234e-17,-0.731849,-0.681466,Frank Foster,51.349998,0
